# Exploração da API do Comex Stat (MDIC)

Objetivo: identificar como obter os valores mensais de:
- Exportações (US$ bi)
- Importações (US$ bi)
- Saldo Comercial (US$ bi)

Fontes candidatas:
- API oficial: https://api-comexstat.mdic.gov.br/
- Alternativa: BCB (SGS) — séries 22707, 22708, 22709

In [ ]:
import requests

# Testar a API do Comex Stat — endpoint de metadados
URL_COMEX = "https://api-comexstat.mdic.gov.br/general/details"

# Primeiro, uma chamada OPTIONS para ver se o endpoint existe
r = requests.options(URL_COMEX, timeout=10)
print(f"OPTIONS — Status: {r.status_code}")
print(f"Headers: {dict(r.headers)}")
print()

# Uma chamada GET simples
r = requests.get(URL_COMEX, timeout=10)
print(f"GET — Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print()
print(r.text[:1000])

In [ ]:
import requests

URL_COMEX = "https://api-comexstat.mdic.gov.br"

r = requests.get(
    URL_COMEX,
    timeout=10
)

print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print(r.text[:2000])

In [ ]:
# Testar POST /general para exportações
URL_GENERAL = "https://api-comexstat.mdic.gov.br/general"

body = {
    "flow": "export",
    "monthDetail": True,
    "period": {"from": "2026-06", "to": "2026-07"},
    "filters": [],
    "details": [],
    "metrics": ["metricFOB"],
}

r = requests.post(URL_GENERAL, json=body, timeout=20)
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print()

if r.status_code == 200:
    dados = r.json()
    print(f"Chaves do retorno: {list(dados.keys())}")
    print()
    print("Conteúdo (primeiras 2000 chars):")
    print(json.dumps(dados, indent=2, ensure_ascii=False)[:2000])
else:
    print(f"Erro: {r.text[:500]}")

In [ ]:
# Ver o retorno sem formatar
print("Tipo:", type(r.json()))
print()
print("Chaves e tipos dos valores:")
for chave, valor in r.json().items():
    print(f"  {chave!r}: {type(valor).__name__} → {repr(valor)[:200]}")

In [ ]:
# Testar com período mais antigo (2024)
body = {
    "flow": "export",
    "monthDetail": True,
    "period": {"from": "2024-01", "to": "2024-03"},
    "filters": [],
    "details": [],
    "metrics": ["metricFOB"],
}

r = requests.post(URL_GENERAL, json=body, timeout=20)
print(f"Status: {r.status_code}")
dados = r.json()

print(f"\nChave 'data': {type(dados.get('data'))}")
print(f"Conteúdo de 'data': {repr(dados.get('data'))[:1500]}")

In [ ]:
# Testar com período bem amplo
body = {
    "flow": "export",
    "monthDetail": False,  # False = agrega por ano
    "period": {"from": "2024-01", "to": "2024-12"},
    "filters": [],
    "details": [],
    "metrics": ["metricFOB"],
}

r = requests.post(URL_GENERAL, json=body, timeout=20)
dados = r.json()
print(f"Status: {r.status_code}")
print(f"data: {repr(dados.get('data'))[:1500]}")

In [ ]:
dados = r.json()
print("Mensagem:", dados.get("message"))
print("Sucesso:", dados.get("success"))
print("Processo:", dados.get("processo_info"))